# TP 3 / 10 — GLM logistique (cours de tarification)

**M2 Actuariat — Abidjan**

---

## Objectifs
- Ajuster un **GLM Bernoulli – lien logit** sur les données préparées au TP 2.
- Lire et **interpréter** la sortie `statsmodels` (coefficients, p-values, IC, déviance, AIC).
- Calculer les **odds ratios** et leur **intervalle de confiance**, et les lire avec l'œil d'un actuaire.
- Faire une **sélection de variables** : par p-value puis par **backward AIC**.
- Diagnostiquer la **multicolinéarité** avec le **VIF**.

> Lien avec le cours de tarification : la régression logistique est un GLM avec loi de Bernoulli et lien logit. C'est l'équivalent « probabilité de sinistre » du modèle de fréquence (qui, lui, utilise une loi de Poisson). Tout ce qu'on fait ici se transpose mot pour mot au cas Poisson — il suffit de changer `Logit` en `Poisson`.

**Durée estimée : ~45 min.**

---


## 0. Imports et chargement des jeux préparés au TP 2


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

X_train = pd.read_csv("X_train.csv")
X_test  = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test  = pd.read_csv("y_test.csv").squeeze("columns")

# Toutes les colonnes doivent être numériques (issues du one-hot et de la standardisation du TP 2)
X_train = X_train.astype(float)
X_test  = X_test.astype(float)

# --- Nettoyage indispensable avant un GLM ---
# 1) Colonnes (quasi-)constantes sur le train : aucune information, font planter le Hessien.
std_train = X_train.std(axis=0)
cols_const = std_train[std_train < 1e-8].index.tolist()
if cols_const:
    print(f"Colonnes constantes retirées : {cols_const}")
    X_train = X_train.drop(columns=cols_const)
    X_test  = X_test.drop(columns=cols_const)

# 2) Colonnes linéairement dépendantes (piège des dummies sans drop_first).
#    On utilise une décomposition QR pour repérer un sous-ensemble de colonnes de plein rang.
M = np.column_stack([np.ones(len(X_train)), X_train.values])  # on inclut la constante
_, R = np.linalg.qr(M)
diag = np.abs(np.diag(R))
tol = diag.max() * 1e-10
keep_mask = diag[1:] > tol  # on saute la 1ère colonne (constante)
cols_dep = X_train.columns[~keep_mask].tolist()
if cols_dep:
    print(f"Colonnes linéairement dépendantes retirées : {cols_dep}")
    X_train = X_train.drop(columns=cols_dep)
    X_test  = X_test.drop(columns=cols_dep)

print(f"\nX_train : {X_train.shape}, y_train : {y_train.shape}, taux BAD = {y_train.mean():.3%}")
print(f"X_test  : {X_test.shape},  y_test  : {y_test.shape},  taux BAD = {y_test.mean():.3%}")
print(f"\nNombre de variables explicatives après nettoyage : {X_train.shape[1]}")

## 1. Rappel théorique éclair

Soit $Y_i \in \{0,1\}$ la cible (1 si `BAD`). On suppose :

$$ Y_i \mid X_i \sim \mathrm{Bernoulli}(\pi_i), \qquad \mathrm{logit}(\pi_i) = \log\frac{\pi_i}{1-\pi_i} = \beta_0 + \beta^\top X_i. $$

L'estimation se fait par **maximum de vraisemblance** (IRLS sous le capot de `statsmodels`). L'objet renvoyé contient :
- les estimateurs $\hat\beta$ et leur erreur-type,
- les p-values du test de Wald $H_0: \beta_j = 0$,
- la **déviance** $D = -2 \log L$ (analogue du RSS),
- les critères d'information **AIC** $= D + 2k$, **BIC** $= D + k\log n$.

L'**odds ratio** $e^{\beta_j}$ se lit ainsi : « toutes choses égales par ailleurs, augmenter $X_j$ d'une unité multiplie la **cote** $\pi/(1-\pi)$ par $e^{\beta_j}$ ».

> Attention : nos variables numériques sont **standardisées**. Une « unité » signifie donc **un écart-type** sur le train (≈ 14 ans pour `DrivAge`).


## 2. Premier GLM : toutes les variables


In [ ]:
# Ajout de la constante
X_train_const = sm.add_constant(X_train, has_constant="add")
X_test_const  = sm.add_constant(X_test,  has_constant="add")

# Ajustement
# method='bfgs' : optimisation quasi-Newton sans inversion explicite du Hessien,
# beaucoup plus robuste que Newton-Raphson par d\u00e9faut sur des matrices de design \u00e0 nombreux one-hot.
model_full = sm.Logit(y_train, X_train_const).fit(disp=0, maxiter=500, method="bfgs")

print(model_full.summary())

In [ ]:
# Indicateurs synthétiques utiles
print(f"Log-vraisemblance        : {model_full.llf:.1f}")
print(f"Log-vraisemblance du nul : {model_full.llnull:.1f}")
print(f"Pseudo R² de McFadden    : {model_full.prsquared:.4f}")
print(f"AIC                       : {model_full.aic:.1f}")
print(f"BIC                       : {model_full.bic:.1f}")
print(f"Nombre de paramètres      : {len(model_full.params)}")


### Question 1
- Combien de coefficients ont été estimés ? Combien sont significatifs au seuil 5% ?
- Le pseudo-R² de McFadden vaut ~0.05–0.10. Un actuaire dirait-il que c'est « peu » ? Comparez avec un R² classique en régression linéaire : pourquoi les ordres de grandeur sont-ils si différents ?
- Que représente concrètement l'**intercept** $\beta_0$ dans ce modèle, sachant que les variables numériques sont centrées-réduites ?

*Votre réponse :*


## 3. Coefficients et odds ratios

C'est la lecture « actuarielle » du modèle.


In [ ]:
def coef_table(result, alpha=0.05):
    conf = result.conf_int(alpha=alpha)
    tab = pd.DataFrame({
        "coef":      result.params,
        "std_err":   result.bse,
        "p_value":   result.pvalues,
        "OR":        np.exp(result.params),
        "OR_low":    np.exp(conf[0]),
        "OR_high":   np.exp(conf[1]),
    })
    tab["significatif"] = tab["p_value"] < alpha
    return tab.sort_values("p_value")

tab = coef_table(model_full)
tab.head(25)


In [ ]:
# Visualisation : odds ratios significatifs avec leur IC 95%
sig = tab[(tab["significatif"]) & (tab.index != "const")].copy()
sig["abs_log_OR"] = np.abs(np.log(sig["OR"]))
sig = sig.sort_values("abs_log_OR")

fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(sig))))
y_pos = np.arange(len(sig))
ax.errorbar(sig["OR"], y_pos,
            xerr=[sig["OR"] - sig["OR_low"], sig["OR_high"] - sig["OR"]],
            fmt="o", color="steelblue", ecolor="gray", capsize=3)
ax.axvline(1, color="red", linestyle="--", label="OR = 1 (pas d'effet)")
ax.set_yticks(y_pos)
ax.set_yticklabels(sig.index)
ax.set_xscale("log")
ax.set_xlabel("Odds ratio (échelle log)")
ax.set_title("Effets significatifs (5%) sur le risque de sinistre")
ax.legend()
plt.tight_layout()
plt.show()


### Question 2 — interprétation actuarielle
- Quel est l'odds ratio de `BonusMalus` ? Une variation d'**un écart-type** de `BonusMalus` multiplie la cote de sinistre par combien ? Cela vous semble-t-il cohérent avec la mécanique du bonus-malus français et avec le diagnostic du TP 1 (effet concentré sur la queue BM > 100) ?
- **Quel est le signe** du coefficient de `DrivAge` dans ce GLM ? Vous attendiez-vous à ce signe au vu du TP 1 ? Comment l'expliquer compte tenu du **VIF élevé** entre `DrivAge` et `LicAge` (Q4) et du signe du coefficient de `LicAge` ?
- Identifiez **deux modalités** de `VehBody` qui apparaissent comme significativement plus risquées que la **modalité de référence** (celle qui a été retirée par `drop_first=True`). Quelle est cette modalité de référence dans notre cas, et est-elle un bon choix ? Que proposeriez-vous pour rendre la lecture des odds ratios plus interprétable ?

*Votre réponse :*


## 4. Sélection de variables (1) — par p-value

Approche simple : on garde les variables significatives au seuil 5%, puis on refit.

⚠️ **Avertissement** : cette stratégie est très utilisée mais a des défauts (multiplicité des tests, instabilité). On la compare ensuite à une **backward AIC**.


In [ ]:
ALPHA = 0.05
vars_sig = tab[(tab["p_value"] < ALPHA) & (tab.index != "const")].index.tolist()
print(f"{len(vars_sig)} variables significatives au seuil {ALPHA}")
print(vars_sig)

X_train_sig = sm.add_constant(X_train[vars_sig], has_constant="add")
model_sig = sm.Logit(y_train, X_train_sig).fit(disp=0, maxiter=500, method="bfgs")
print(f"\nAIC mod\u00e8le complet     : {model_full.aic:.1f}")
print(f"AIC modèle 'p < {ALPHA}' : {model_sig.aic:.1f}")
print(f"Log-vraisemblance       : {model_sig.llf:.1f} vs {model_full.llf:.1f}")


## 5. Sélection de variables (2) — backward AIC

**Idée naïve.** On part du modèle complet ; à chaque itération on essaie de retirer **une** variable, on garde le modèle qui minimise l'AIC ; on s'arrête quand plus aucune suppression ne fait baisser l'AIC. Coût : $O(p^2)$ fits — vite prohibitif.

**Idée rapide.** On peut s'éviter $p$ fits par itération grâce au résultat classique :

$$\Delta\mathrm{AIC} \;\approx\; 2 - z_j^2 \qquad \text{en retirant la variable } j,$$

où $z_j = \hat\beta_j / \widehat{se}(\hat\beta_j)$ est la statistique de **Wald**. Conséquence :

$$\Delta\mathrm{AIC} < 0 \;\iff\; z_j^2 > 2 \;\iff\; |z_j| > \sqrt{2} \approx 1{,}41.$$

> En pratique : à chaque itération on retire la variable au plus petit $|z|$ si $z^2 < 2$, on refit, on recommence. **Complexité $O(p)$ au lieu de $O(p^2)$**.
>
> *(Attention : c'est une approximation au 2nd ordre du log-vraisemblance. Pour des $z$ très proches de $\sqrt 2$ ou en présence de fortes corrélations résiduelles, l'algorithme exact peut donner un résultat légèrement différent.)*

> En théorie de la sélection de modèles, **l'AIC est cohérent avec la minimisation de la divergence de Kullback-Leibler** entre le vrai modèle et le modèle ajusté. Ce n'est donc pas un simple critère ad hoc.

In [ ]:
def _fit_logit(y, X):
    return sm.Logit(y, X).fit(disp=0, maxiter=500, method="bfgs")

def backward_aic_fast(X, y, verbose=False):
    """Backward AIC accélérée par approximation de Wald.

    Résultat classique : retirer la variable j d'un GLM modifie l'AIC de
        ΔAIC ≈ 2 - z_j²
    où z_j = β̂_j / se(β̂_j) est la statistique de Wald.
    Donc ΔAIC < 0  ⇔  z_j² > 2  ⇔  |z_j| > √2 ≈ 1.41.

    À chaque itération on retire UNE seule variable : celle au plus petit |z|
    si z² < 2. On refit, on recommence. Complexité O(p) au lieu de O(p²).
    """
    import time
    cols = list(X.columns)
    t0 = time.time()
    iter_idx = 0
    while len(cols) > 1:
        m = _fit_logit(y, sm.add_constant(X[cols], has_constant="add"))
        z2 = (m.tvalues.drop("const", errors="ignore")) ** 2
        worst, worst_z2 = z2.idxmin(), float(z2.min())
        if worst_z2 >= 2.0:
            break  # plus aucune suppression ne fait baisser l'AIC
        cols.remove(worst)
        iter_idx += 1
        if verbose:
            print(f"[{iter_idx:2d}] Retire {worst:35s}  z²={worst_z2:5.2f}  → {len(cols)} vars  AIC={m.aic:.1f}")
    final = _fit_logit(y, sm.add_constant(X[cols], has_constant="add"))
    if verbose:
        print(f"\nDurée : {time.time()-t0:.1f} s — {iter_idx} itérations")
    return cols, final.aic

print("Démarrage de la backward AIC (version rapide, O(p) fits)...")
selected_vars, final_aic = backward_aic_fast(X_train, y_train, verbose=True)

print(f"\n{len(selected_vars)} variables retenues, AIC final = {final_aic:.2f}")
print("Variables retenues :", selected_vars)

In [ ]:
X_train_bw = sm.add_constant(X_train[selected_vars], has_constant="add")
model_bw = sm.Logit(y_train, X_train_bw).fit(disp=0, maxiter=500, method="bfgs")

print(f"AIC complet           : {model_full.aic:.1f}  ({len(model_full.params)} params)")
print(f"AIC sélection p-value : {model_sig.aic:.1f}  ({len(model_sig.params)} params)")
print(f"AIC backward AIC      : {model_bw.aic:.1f}  ({len(model_bw.params)} params)")


### Question 3
- La sélection par p-value et la backward AIC retiennent-elles **exactement** les mêmes variables ? Donnez un exemple où elles divergent.
- Pourquoi la **backward AIC** est-elle généralement préférée par les statisticiens ? Quel est son principal défaut ?
- Si on remplaçait l'AIC par le **BIC**, le modèle final serait-il **plus** ou **moins** parcimonieux ? Pourquoi ? (Indice : $\log n$ vs $2$ avec $n = $ ?)

*Votre réponse :*


## 6. Diagnostic de multicolinéarité — VIF

Le **Variance Inflation Factor** mesure de combien la variance d'un coefficient est gonflée par les corrélations avec les autres variables. Règle empirique :
- VIF < 5 : OK
- 5 ≤ VIF < 10 : à surveiller
- VIF ≥ 10 : multicolinéarité sévère → action requise


In [ ]:
# On calcule le VIF sur les variables numériques + binaires (pas sur les one-hot pour rester lisible)
num_like = ["LicAge", "DrivAge", "BonusMalus", "RiskVar", "HasKmLimit",
            "VehAge_num", "VehMaxSpeed_num", "Gender_F", "MariAlone"]
num_like = [c for c in num_like if c in X_train.columns]

Xv = X_train[num_like].values
vif = pd.DataFrame({
    "variable": num_like,
    "VIF": [variance_inflation_factor(Xv, i) for i in range(Xv.shape[1])],
}).sort_values("VIF", ascending=False)
print(vif.to_string(index=False))


### Question 4
- Quelle(s) variable(s) tombent dans la zone **« à surveiller »** (5 ≤ VIF < 10) ? Aucune n'est en zone « sévère » (≥ 10) — comment situeriez-vous donc la gravité du problème ici ?
- Proposez **deux stratégies** concrètes pour réduire la colinéarité (sans simplement supprimer une variable). Indice : on peut **transformer** ou **combiner** des variables.
- En pratique en tarification, est-il fréquent d'avoir des variables très corrélées ? Donnez un exemple.

*Votre réponse :*


## 7. Synthèse et préparation du TP 4

Nous avons trois modèles candidats :
- `model_full` : 50+ variables, modèle de référence.
- `model_sig` : variables p < 5%, économe et simple.
- `model_bw`  : sélection backward AIC, optimal au sens AIC.

Pour le TP 4 (évaluation), nous garderons le **modèle backward AIC** car il offre le meilleur compromis biais–variance d'après l'AIC. Mais l'idée du TP 5 sera justement de **comparer plusieurs modèles** sur des métriques **actuarielles** (lift, Gini), qui peuvent désigner un vainqueur différent !

### Question 5 — synthèse
- En tarification, on tarife des **groupes** d'assurés (cellules) et non des individus. En quoi la lecture du tableau des odds ratios (question 2) sert-elle directement à construire une grille tarifaire ?
- Si demain votre direction veut un modèle « explicable au régulateur », lequel des trois choisiriez-vous ? Pourquoi ?
- Quelles **deux variables** sont selon vous les plus utiles métier, indépendamment de leur poids statistique ? (Argumentez avec ce que vous savez du métier.)

*Votre réponse :*


In [ ]:
# Sauvegarde du modèle backward et des variables sélectionnées pour les TP suivants
import pickle
with open("model_glm_bw.pkl", "wb") as f:
    pickle.dump({"model": model_bw, "vars": selected_vars}, f)

# On enregistre aussi des prédictions probabilistes prêtes à servir au TP 4
proba_train = model_bw.predict(X_train_bw)
proba_test  = model_bw.predict(sm.add_constant(X_test[selected_vars], has_constant="add"))

pd.DataFrame({"proba": proba_train}).to_csv("proba_glm_train.csv", index=False)
pd.DataFrame({"proba": proba_test }).to_csv("proba_glm_test.csv",  index=False)

print("Sauvegardes : model_glm_bw.pkl, proba_glm_train.csv, proba_glm_test.csv")
print(f"\nAperçu des probas test : min={proba_test.min():.3f}, max={proba_test.max():.3f}, moy={proba_test.mean():.3f}")


---
**Prochain TP : TP 4 — Évaluation actuarielle (lift, Gini, calibration).**

On chargera `proba_glm_test.csv` et on construira les courbes que tout actuaire connaît : la courbe de lift, la courbe de Lorenz et l'indice de Gini.
